In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Interactive k-NN Example 2 (Manhattan Distance)

This notebook mirrors Lecture 1 / Example 2 and shows how an outlier can change k-NN predictions.


In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets

base_points = [
    ("A", 1.2, 2.8, 0, False),
    ("B", 3.7, 4.1, 0, False),
    ("C", 6.2, 5.0, 1, False),
    ("D", 7.5, 3.8, 1, False),
    ("E", 7.2, 4.7, 0, True),
]

def classify(new_x, new_y, k, include_outlier):
    points = [p for p in base_points if include_outlier or not p[4]]
    distances = []
    for name, x, y, cls, outlier in points:
        d = abs(new_x - x) + abs(new_y - y)
        distances.append((name, x, y, cls, outlier, d))
    distances.sort(key=lambda t: t[5])
    neighbors = distances[: max(1, min(k, len(distances)))]
    c0 = sum(1 for n in neighbors if n[3] == 0)
    c1 = sum(1 for n in neighbors if n[3] == 1)
    pred = 1 if c1 > c0 else 0
    if c0 == c1:
        pred = neighbors[0][3]
    return points, neighbors, pred

def plot_knn_example2(new_x=7.5, new_y=4.5, k=3, include_outlier=True):
    points, neighbors, pred = classify(new_x, new_y, k, include_outlier)
    fig, ax = plt.subplots(figsize=(8, 5.6))

    for name, x, y, cls, outlier in points:
        color = "#b45309" if outlier else ("#1d4ed8" if cls == 0 else "#dc2626")
        ax.scatter(x, y, color=color, marker="x", s=110)
        label = f"{name} (outlier)" if outlier else name
        ax.text(x + 0.07, y + 0.07, label)

    for _, x, y, _, _, _ in neighbors:
        ax.plot([new_x, x], [new_y, y], color="gray", linewidth=1)

    ax.scatter(new_x, new_y, color="#7e22ce", marker="x", s=130)
    ax.text(new_x + 0.07, new_y + 0.07, "F")

    ax.set_xlim(0, 8)
    ax.set_ylim(2, 6)
    ax.set_xlabel("X-axis")
    ax.set_ylabel("Y-axis")
    ax.grid(alpha=0.25)
    pred_text = "Class 1" if pred == 1 else "Class 0"
    ax.set_title(f"Prediction: {pred_text} | k={k} | outlier {'on' if include_outlier else 'off'}")
    plt.show()

widgets.interact(
    plot_knn_example2,
    new_x=widgets.FloatSlider(min=0, max=8, step=0.1, value=7.5),
    new_y=widgets.FloatSlider(min=2, max=6, step=0.1, value=4.5),
    k=widgets.IntSlider(min=1, max=5, step=1, value=3),
    include_outlier=True,
)
